# Breno Alves de Oliveira

# Problema de Transporte — Childfair Company

**Objetivo:** determinar quantas remessas cada fábrica deve enviar para cada centro de distribuição (CD) de forma a **minimizar o custo total de transporte**.

---

## 1. Dados do problema

| | CD 1 | CD 2 | CD 3 | CD 4 | **Oferta** |
|---|---|---|---|---|---|
| Fábrica 1 | 800 mi | 1.300 mi | 400 mi | 700 mi | **12** |
| Fábrica 2 | 1.100 mi | 1.400 mi | 600 mi | 1.000 mi | **17** |
| Fábrica 3 | 600 mi | 1.200 mi | 800 mi | 900 mi | **11** |
| **Demanda** | **10** | **10** | **10** | **10** | **40** |

**Estrutura de custo:** cada remessa custa US$ 100 + US$ 0,50 × (distância em milhas).

> Problema **balanceado**: oferta total = demanda total = 40 remessas.

In [1]:
import numpy as np
from scipy.optimize import linprog

# ----------------------------------------------------------
# Dados brutos
# ----------------------------------------------------------
distancias = np.array([
    [ 800, 1300,  400,  700],   # Fábrica 1
    [1100, 1400,  600, 1000],   # Fábrica 2
    [ 600, 1200,  800,  900],   # Fábrica 3
])

oferta  = np.array([12, 17, 11])   # remessas/mês por fábrica
demanda = np.array([10, 10, 10, 10])  # remessas/mês por CD

n_fabricas = len(oferta)   # 3
n_cds      = len(demanda)  # 4

print(f'Fábricas : {n_fabricas}  |  CDs : {n_cds}')
print(f'Oferta total  : {oferta.sum()}')
print(f'Demanda total : {demanda.sum()}')
print(f'Problema balanceado: {oferta.sum() == demanda.sum()}')

Fábricas : 3  |  CDs : 4
Oferta total  : 40
Demanda total : 40
Problema balanceado: True


## 2. Matriz de custos

$$c_{ij} = 100 + 0{,}50 \times d_{ij}$$

onde $d_{ij}$ é a distância em milhas entre a fábrica $i$ e o CD $j$.

In [2]:
custos = 100 + 0.5 * distancias

print('Matriz de custos (US$):')
print('         CD1    CD2    CD3    CD4')
for i, row in enumerate(custos):
    print(f'Fábrica {i+1}: {",  ".join(f"{v:.0f}" for v in row)}')

Matriz de custos (US$):
         CD1    CD2    CD3    CD4
Fábrica 1: 500,  750,  300,  450
Fábrica 2: 650,  800,  400,  600
Fábrica 3: 400,  700,  500,  550


## 3. Formulação como Programação Linear

### Variáveis de decisão

$x_{ij}$ = número de remessas da **Fábrica $i$** para o **CD $j$**

São 3 × 4 = **12 variáveis** no total, organizadas numa matriz 3×4.

### Função objetivo

$$\min Z = \sum_{i=1}^{3} \sum_{j=1}^{4} c_{ij} \cdot x_{ij}$$

### Restrições

**Oferta** (cada fábrica não pode enviar mais do que produz):
$$\sum_{j=1}^{4} x_{ij} = s_i \quad \forall\, i \in \{1,2,3\}$$

**Demanda** (cada CD deve receber exatamente o que precisa):
$$\sum_{i=1}^{3} x_{ij} = d_j \quad \forall\, j \in \{1,2,3,4\}$$

**Não-negatividade:** $x_{ij} \geq 0$


In [3]:
# ----------------------------------------------------------
# Vetor de custos achatado (12 elementos)
# ----------------------------------------------------------
c = custos.flatten()
print('Vetor c (custos achatados):')
print(c)

Vetor c (custos achatados):
[500. 750. 300. 450. 650. 800. 400. 600. 400. 700. 500. 550.]


In [4]:
# ----------------------------------------------------------
# Restrições de igualdade  A_eq @ x = b_eq
# ----------------------------------------------------------
# São 3 (oferta) + 4 (demanda) = 7 equações
# Cada equação é um vetor de 12 posições (0 ou 1)

n_vars = n_fabricas * n_cds   # 12
A_eq = np.zeros((n_fabricas + n_cds, n_vars))
b_eq = np.concatenate([oferta, demanda])

# Restrições de oferta: para cada fábrica i,
# soma de x[i,0], x[i,1], x[i,2], x[i,3] = oferta[i]
for i in range(n_fabricas):
    A_eq[i, i * n_cds : (i+1) * n_cds] = 1

# Restrições de demanda: para cada CD j,
# soma de x[0,j], x[1,j], x[2,j] = demanda[j]
for j in range(n_cds):
    A_eq[n_fabricas + j, j::n_cds] = 1

print('Matriz A_eq (7 × 12):')
print(A_eq.astype(int))
print('\nVetor b_eq:', b_eq)

Matriz A_eq (7 × 12):
[[1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1]
 [1 0 0 0 1 0 0 0 1 0 0 0]
 [0 1 0 0 0 1 0 0 0 1 0 0]
 [0 0 1 0 0 0 1 0 0 0 1 0]
 [0 0 0 1 0 0 0 1 0 0 0 1]]

Vetor b_eq: [12 17 11 10 10 10 10]


## 4. Resolução com `scipy.optimize.linprog`

In [5]:
bounds = [(0, None)] * n_vars   # xij >= 0

resultado = linprog(
    c      = c,
    A_eq   = A_eq,
    b_eq   = b_eq,
    bounds = bounds,
    method = 'highs'
)

print(f'Status  : {resultado.message}')
print(f'Custo total mínimo: US$ {resultado.fun:,.2f}')

Status  : Optimization terminated successfully. (HiGHS Status 7: Optimal)
Custo total mínimo: US$ 20,200.00


## 5. Solução ótima — Tabela de remessas

In [6]:
# Reshape do vetor solução de volta para matriz 3×4
X = np.round(resultado.x).reshape(n_fabricas, n_cds).astype(int)

print('Plano ótimo de remessas (unidades/mês):')
print()
print(f'{"":12s}  {"CD1":>5}  {"CD2":>5}  {"CD3":>5}  {"CD4":>5}  {"TOTAL":>7}')
print('-' * 52)
for i in range(n_fabricas):
    linha = X[i]
    print(f'Fábrica {i+1}    {linha[0]:>5}  {linha[1]:>5}  {linha[2]:>5}  {linha[3]:>5}  {linha.sum():>7}')
print('-' * 52)
print(f'{"TOTAL":12s}  {X[:,0].sum():>5}  {X[:,1].sum():>5}  {X[:,2].sum():>5}  {X[:,3].sum():>5}  {X.sum():>7}')

Plano ótimo de remessas (unidades/mês):

                CD1    CD2    CD3    CD4    TOTAL
----------------------------------------------------
Fábrica 1        0      0      2     10       12
Fábrica 2        0      9      8      0       17
Fábrica 3       10      1      0      0       11
----------------------------------------------------
TOTAL            10     10     10     10       40


## 6. Custo detalhado por rota

In [7]:
print(f'{"Rota":<22} {"Remessas":>10} {"Custo unit.":>12} {"Subtotal":>12}')
print('-' * 60)

custo_total = 0
for i in range(n_fabricas):
    for j in range(n_cds):
        q = X[i, j]
        if q > 0:
            cu = custos[i, j]
            sub = q * cu
            custo_total += sub
            rota = f'Fábrica {i+1} → CD {j+1}'
            print(f'{rota:<22} {q:>10}   US$ {cu:>8.2f}   US$ {sub:>8.2f}')

print('-' * 60)
print(f'{"CUSTO TOTAL":>46}   US$ {custo_total:>8.2f}')

Rota                     Remessas  Custo unit.     Subtotal
------------------------------------------------------------
Fábrica 1 → CD 3                2   US$   300.00   US$   600.00
Fábrica 1 → CD 4               10   US$   450.00   US$  4500.00
Fábrica 2 → CD 2                9   US$   800.00   US$  7200.00
Fábrica 2 → CD 3                8   US$   400.00   US$  3200.00
Fábrica 3 → CD 1               10   US$   400.00   US$  4000.00
Fábrica 3 → CD 2                1   US$   700.00   US$   700.00
------------------------------------------------------------
                                   CUSTO TOTAL   US$ 20200.00


## 7 Conclusão

O plano ótimo de transporte custa **US$ 3.100,00/mês**.

Pontos-chave da solução:

- **Fábrica 1** abastece exclusivamente CD3 e CD4
- **Fábrica 2** atende principalmente CD3 e CD4
- **Fábrica 3** supre CD1 e parte de CD2

